In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable
from datetime import datetime
import pytz

# 1. Parámetros y Configuración
dbutils.widgets.removeAll()
dbutils.widgets.dropdown("env", "dev", ["dev", "prod"], "Ambiente")
dbutils.widgets.text("process_date", "01012026", "Fecha Proceso (DDMMYYYY)")

In [0]:
env = dbutils.widgets.get("env")
process_date = dbutils.widgets.get("process_date")
catalog = f"{env}_fraud"

# Configuración Hora Perú para auditoría
peru_tz = pytz.timezone('America/Lima')
ts_peru = datetime.now(peru_tz).strftime('%Y-%m-%d %H:%M:%S')

print(f"🚀 Iniciando carga de Dimensiones Gold para el catálogo: {catalog}\n")

# --- DIMENSIÓN 1: Clientes ---
print("🔄 Procesando dim_clientes...")
df_cust_silver = spark.table(f"{catalog}.silver.clientes_silver")

df_dim_cust = df_cust_silver.select(
    "id_cliente", "nombre", "rango_edad", "rango_fico", "estado", "ingreso_anual"
).dropDuplicates(["id_cliente"])

dt_cust = DeltaTable.forName(spark, f"{catalog}.gold.dim_clientes")
dt_cust.alias("t").merge(
    df_dim_cust.alias("s"), 
    "t.id_cliente = s.id_cliente"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()


# --- DIMENSIÓN 2: Tarjetas ---
print("🔄 Procesando dim_tarjetas...")
df_cards_silver = spark.table(f"{catalog}.silver.tarjetas_silver")

df_dim_cards = df_cards_silver.select(
    "id_tarjeta", "tipo_tarjeta", "marca_tarjeta", "limite_credito"
).dropDuplicates(["id_tarjeta"])

dt_cards = DeltaTable.forName(spark, f"{catalog}.gold.dim_tarjetas")
dt_cards.alias("t").merge(
    df_dim_cards.alias("s"), 
    "t.id_tarjeta = s.id_tarjeta"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
# --- DIMENSIÓN 3: MCC (Catálogo de Comercios) ---
print("🔄 Procesando dim_mcc...")
df_mcc_silver = spark.table(f"{catalog}.silver.mcc_silver")

df_dim_mcc = df_mcc_silver.select(
    "codigo_mcc", "descripcion", "categoria_grupo"
).dropDuplicates(["codigo_mcc"])

dt_mcc = DeltaTable.forName(spark, f"{catalog}.gold.dim_mcc")
dt_mcc.alias("t").merge(
    df_dim_mcc.alias("s"), 
    "t.codigo_mcc = s.codigo_mcc"
).whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()

# --- RESUMEN FINAL ---
print("\n" + "═" * 60)
print(" ✅ CARGA DE DIMENSIONES COMPLETADA ".center(60, "═"))
print("═" * 60)
print(f"dim_clientes : {spark.table(f'{catalog}.gold.dim_clientes').count():,} registros")
print(f"dim_tarjetas : {spark.table(f'{catalog}.gold.dim_tarjetas').count():,} registros")
print(f"dim_mcc      : {spark.table(f'{catalog}.gold.dim_mcc').count():,} registros")
print("═" * 60)

display(spark.table(f"{catalog}.gold.dim_clientes").limit(3))